In [23]:
!pip install kaggle pandas numpy scikit-learn xgboost joblib -q

In [24]:
import kagglehub
kagglehub.login()

Kaggle credentials set.
Kaggle credentials successfully validated.


In [31]:
!kaggle datasets download -d kaushil268/disease-prediction-using-machine-learning
!unzip -o disease-prediction-using-machine-learning.zip

!kaggle datasets download -d itachi9604/disease-symptom-description-dataset
!unzip -o disease-symptom-description-dataset.zip

Dataset URL: https://www.kaggle.com/datasets/kaushil268/disease-prediction-using-machine-learning
License(s): DbCL-1.0
disease-prediction-using-machine-learning.zip: Skipping, found more recently modified local copy (use --force to force download)
Archive:  disease-prediction-using-machine-learning.zip
  inflating: Testing.csv             
  inflating: Training.csv            
Dataset URL: https://www.kaggle.com/datasets/itachi9604/disease-symptom-description-dataset
License(s): CC-BY-SA-4.0
disease-symptom-description-dataset.zip: Skipping, found more recently modified local copy (use --force to force download)
Archive:  disease-symptom-description-dataset.zip
  inflating: Symptom-severity.csv    
  inflating: dataset.csv             
  inflating: symptom_Description.csv  
  inflating: symptom_precaution.csv  


In [38]:
import pandas as pd
import numpy as np
import random
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report

from xgboost import XGBClassifier

In [39]:
df = pd.read_csv("Training.csv")
df = df.loc[:, ~df.columns.str.contains('^Unnamed')]

# CLEAN LABELS (IMPORTANT)
df["prognosis"] = df["prognosis"].str.strip()

X = df.drop("prognosis", axis=1)
y = df["prognosis"]

symptom_cols = X.columns.tolist()

In [40]:
le = LabelEncoder()
y_encoded = le.fit_transform(y)

In [41]:
def inject_noise(X, drop_prob=0.15, flip_prob=0.03):
    Xn = X.copy().values

    for i in range(Xn.shape[0]):
        for j in range(Xn.shape[1]):

            # drop true symptom
            if Xn[i][j] == 1 and random.random() < drop_prob:
                Xn[i][j] = 0

            # add false symptom
            elif Xn[i][j] == 0 and random.random() < flip_prob:
                Xn[i][j] = 1

    return pd.DataFrame(Xn, columns=symptom_cols)

X_noisy = inject_noise(X)

# combine clean + noisy
X_full = pd.concat([X, X_noisy], ignore_index=True)
y_full = np.concatenate([y_encoded, y_encoded])

In [42]:
X_train, X_val, y_train, y_val = train_test_split(
    X_full,
    y_full,
    test_size=0.2,
    random_state=42,
    stratify=y_full
)

In [43]:
model = XGBClassifier(
    n_estimators=350,
    max_depth=6,              # ↓ reduces overfitting
    learning_rate=0.05,
    subsample=0.8,            # ↓ randomness
    colsample_bytree=0.8,
    reg_lambda=2,             # L2 regularization
    reg_alpha=0.5,            # L1 regularization
    eval_metric="mlogloss",
    tree_method="hist",
    random_state=42
)

model.fit(X_train, y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='mlogloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.05, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=6, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=350, n_jobs=None,
              num_parallel_tree=None, ...)

In [44]:
y_pred = model.predict(X_val)

print("Accuracy:", accuracy_score(y_val, y_pred))
print(classification_report(y_val, y_pred))

Accuracy: 0.9908536585365854
              precision    recall  f1-score   support

           0       0.98      0.98      0.98        48
           1       1.00      0.98      0.99        48
           2       0.96      1.00      0.98        48
           3       1.00      1.00      1.00        48
           4       1.00      0.98      0.99        48
           5       1.00      1.00      1.00        48
           6       1.00      1.00      1.00        48
           7       1.00      1.00      1.00        48
           8       1.00      1.00      1.00        48
           9       1.00      0.98      0.99        48
          10       1.00      1.00      1.00        48
          11       1.00      1.00      1.00        48
          12       1.00      1.00      1.00        48
          13       0.98      1.00      0.99        48
          14       0.98      0.98      0.98        48
          15       1.00      0.98      0.99        48
          16       0.98      1.00      0.99        4

In [45]:
desc_df = pd.read_csv("symptom_Description.csv")
prec_df = pd.read_csv("symptom_precaution.csv")

desc_df["Disease"] = desc_df["Disease"].str.strip()
prec_df["Disease"] = prec_df["Disease"].str.strip()

In [46]:
def get_description(disease):
    row = desc_df[desc_df["Disease"] == disease]
    return row["Description"].values[0] if len(row) else "No description"

def get_precautions(disease):
    row = prec_df[prec_df["Disease"] == disease]
    if len(row) == 0:
        return []
    return [p for p in row.values[0][1:] if pd.notna(p)]

In [54]:
import difflib

def normalize_symptom(s, symptom_cols):
    s_clean = s.strip().lower().replace(" ", "_")
    for col in symptom_cols:
        if col.strip().lower() == s_clean:
            return col
    matches = difflib.get_close_matches(s_clean, symptom_cols, n=1, cutoff=0.6)
    return matches[0] if matches else None

def predict_disease(symptoms, min_symptoms=4):
    vec = np.zeros(len(symptom_cols))
    matched = []
    unmatched = []

    for s in symptoms:
        col = normalize_symptom(s, symptom_cols)
        if col:
            vec[symptom_cols.index(col)] = 1
            matched.append(f"{s} → {col}")
        else:
            unmatched.append(s)

    print("Matched:", matched)
    if unmatched:
        print("WARNING - unmatched:", unmatched)

    # ── Filter: not enough symptoms ──────────────────────────────
    if len(matched) < min_symptoms:
        print(f"⚠️  Not enough symptoms ({len(matched)} matched). Please provide at least {min_symptoms}.")
        return []

    vec = vec.reshape(1, -1)
    probs = model.predict_proba(vec)[0]
    top3_idx = np.argsort(probs)[-3:][::-1]

    results = []
    for i in top3_idx:
        disease = le.inverse_transform([i])[0]
        confidence = round(float(probs[i]), 4)
        results.append({
            "disease": disease,
            "confidence": confidence,
            "confidence_label": "High" if confidence > 0.5 else "Medium" if confidence > 0.25 else "Low",
            "description": get_description(disease),
            "precautions": get_precautions(disease)
        })
    return results

In [59]:
# More specific → higher confidence
predict_disease(["fever", "cough", "headache"])

print(predict_disease(["continuous_sneezing", "runny_nose", "cough", "mild_fever", "throat_irritation", "congestion", "chills"]))

print(predict_disease(["fever", "cough", "headache", "breathlessness", "fatigue"]))

print(predict_disease(["fever", "cough", "headache", "breathlessness", "fatigue"]))

print(predict_disease(["itching", "skin_rash", "nodal_skin_eruptions", "dischromic_patches"]))

print(predict_disease(["chest_pain", "vomiting", "breathlessness", "sweating"]))

Matched: ['fever → mild_fever', 'cough → cough', 'headache → headache']
⚠️  Not enough symptoms (3 matched). Please provide at least 4.
Matched: ['continuous_sneezing → continuous_sneezing', 'runny_nose → runny_nose', 'cough → cough', 'mild_fever → mild_fever', 'throat_irritation → throat_irritation', 'congestion → congestion', 'chills → chills']
[{'disease': 'Allergy', 'confidence': 0.7465, 'confidence_label': 'High', 'description': "An allergy is an immune system response to a foreign substance that's not typically harmful to your body.They can include certain foods, pollen, or pet dander. Your immune system's job is to keep you healthy by fighting harmful pathogens.", 'precautions': ['apply calamine', 'cover area with bandage', 'use ice to compress itching']}, {'disease': 'Common Cold', 'confidence': 0.165, 'confidence_label': 'Low', 'description': "The common cold is a viral infection of your nose and throat (upper respiratory tract). It's usually harmless, although it might not fe

In [60]:
joblib.dump(model, "final_nlp_model.pkl")
joblib.dump(le, "label_encoder.pkl")
joblib.dump(symptom_cols, "symptom_list.pkl")

print("✅ EVERYTHING SAVED")

✅ EVERYTHING SAVED
